In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("guns_incident_data_cleaned.csv")
print(df.head())

df = df.drop(columns=["S.No."], errors="ignore")

y = df["Police involvement"] # Target variable
X = df.drop(columns=["Police involvement"]) # Features

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)
print("Final feature count:", X.shape)

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Train Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced"),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        eval_metric="logloss",
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1])
    )
}

results = {}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, prob)
    }

results_df = pd.DataFrame(results).T
print("\n===== MODEL PERFORMANCE =====\n")
print(results_df)

Feature Importance (Permutation Importance)

## Evaluation

In [ ]:
best_model = models["Logistic Regression"]

perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)

sorted_idx = perm.importances_mean.argsort()[::-1]
top_features = X_test.columns[sorted_idx][:15]
top_scores = perm.importances_mean[sorted_idx][:15]

## Visualisation

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_features, top_scores, color="gold")
plt.gca().invert_yaxis()
plt.title("Top Predictors of Police Involvement")
plt.xlabel("Permutation Importance")
plt.show()

## Error analysis by Location Group

In [ ]:
df_test = X_test.copy()
df_test["true"] = y_test
df_test["pred"] = best_model.predict(X_test)

# Find the one-hot encoded "Place of incident" columns
place_cols = df_test.filter(like="Place of incident_").columns

# Assign the most likely location category for each row
df_test["location_group"] = df_test[place_cols].idxmax(axis=1)

error_table = df_test.groupby("location_group").apply(
    lambda g: pd.Series({
        "n": len(g),
        "true_positives": (g["true"] == 1).sum(),
        "false_negatives": ((g["true"] == 1) & (g["pred"] == 0)).sum(),
        "fn_rate": ((g["true"] == 1) & (g["pred"] == 0)).mean()
    })
)

print("\n===== ERROR ANALYSIS BY LOCATION =====\n")
print(error_table)

## Summary: Classification Report

In [ ]:
print("\n===== CLASSIFICATION REPORT (BEST MODEL) =====\n")
print(classification_report(y_test, best_model.predict(X_test)))